In [1]:
import numpy as np
import pandas as pd
import json
import gc
import re
import os
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import random
import torch
# -------------------------------------------------------------------
# Local file path (Windows)
# Make sure this matches the folder where you extracted the dataset
# -------------------------------------------------------------------
DATA_PATH = r"C:\Users\Geeks2_PC10\Documents\Project Sim Dataset\Training Model final"

# Load CSV files
transaction_df = pd.read_csv(os.path.join(DATA_PATH, "transactions_data_south_africa.csv"))
card_df = pd.read_csv(os.path.join(DATA_PATH, "cards_data_south_africa.csv"))
users_df = pd.read_csv(os.path.join(DATA_PATH, "user_data_south_africa.csv"))

# Load MCC JSON file
mcc_series = pd.read_json(os.path.join(DATA_PATH, "mcc_codes.json"), typ='series')
mcc_df = mcc_series.reset_index()
mcc_df.columns = ['mcc_code', 'description']

# Load labels JSON
file_path = os.path.join(DATA_PATH, 'fraud_labels.json')

with open(file_path, 'r') as f:
    raw_json_data = json.load(f)

transaction_labels_dict = raw_json_data['target']

train_fraud_labels = pd.Series(transaction_labels_dict).reset_index()
train_fraud_labels.columns = ['transaction_id', 'is_fraud']
train_fraud_labels['transaction_id'] = pd.to_numeric(train_fraud_labels['transaction_id'])

print("\nAll data files loaded successfully on local computer.")


All data files loaded successfully on local computer.


In [2]:
data_dict = {'transaction': transaction_df,
             'card': card_df,
             'user': users_df,
             'mcc': mcc_df,
             'fraud_labels': train_fraud_labels
            } 
print("\nSuccessfuly save all dataframe in a dictionary called data_dict")


Successfuly save all dataframe in a dictionary called data_dict


In [3]:
# ====================
# Basic prprosessing
# ====================

def cleaned_transaction(df):
    df = df.copy()

    df['date'] = pd.to_datetime(df['date']) # conviert to datetime
    df['errors'] = df['errors'].fillna('no error') # fill nulls
    df['merchant_state'] = np.where(df['merchant_city'] == 'ONLINE', 'ONLINE', df['merchant_state']) # check if city is online and fill state
    df['zip'] = df['zip'] = np.where(df['merchant_city'] == 'ONLINE', 0, df['zip']) # fill zip if city == 'Online' or keep original if false
    df['mcc'] = df['mcc'].fillna(5812) # fill nulls
    return df

# conviert to datetime
data_dict['card']['expires'] = pd.to_datetime(data_dict['card']['expires']) 
data_dict['card']['acct_open_date'] = pd.to_datetime(data_dict['card']['acct_open_date'])
data_dict['transaction']["date"] = pd.to_datetime(data_dict['transaction']["date"])

print("\nSuccessfuly preprocessed")


Successfuly preprocessed


In [4]:
mcc_cl = data_dict['mcc']
transaction_cl = cleaned_transaction(data_dict['transaction'])
card_cl = data_dict['card'].drop(columns=data_dict['card'][['card_on_dark_web']])
user_cl = data_dict['user']
fraud_labels_cl = data_dict['fraud_labels']
print("\nSuccessfuly saved")


Successfuly saved


In [5]:
def column_name_standardization(transaction, card, mcc, user):
    transaction = transaction.rename(columns={'id':'transaction_id'})
    card = card.rename(columns={'id': 'card_id'})
    mcc = mcc.rename(columns={'mcc_code': 'mcc'})
    user = user.rename(columns={'id': 'client_id'})
    return transaction, card, mcc, user
transaction_cl, card_cl, mcc_cl, user_cl = column_name_standardization(transaction_cl, card_cl, mcc_cl, user_cl)

In [6]:
df_merge = transaction_cl.copy()
df_merge = df_merge.merge(card_cl, how='left', on=['card_id','client_id'])
df_merge = df_merge.merge(mcc_cl, how='left', on=['mcc'])
df_merge = df_merge.merge(user_cl, how='left', on=['client_id'])
df_merge = df_merge.merge(fraud_labels_cl, how='left', on=['transaction_id'])

In [7]:
df_merge. columns

Index(['transaction_id', 'date', 'client_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv',
       'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'description', 'current_age', 'retirement_age',
       'birth_year', 'birth_month', 'gender', 'address', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards',
       'is_fraud'],
      dtype='object')

In [7]:
X = df_merge.copy()

In [22]:
# Feature_Engineering.py
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
import warnings

class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom feature engineering transformer for fraud detection.
    - Coerces numeric and date columns to correct dtypes.
    - Handles missing or invalid dates gracefully.
    - Computes expanding statistics per card/client with safety for single rows.
    - Applies one‑hot encoding and label encoding using training mappings.
    - Drops rows with invalid dates only when necessary for rolling windows.
    """

    def __init__(self):
        self.label_encoders = {}          # mappings for high‑cardinality columns
        self.ohe_columns_ = None           # columns produced by one‑hot encoding during fit
        self.numeric_cols_ = None          # columns to be treated as numeric
        self.date_cols_ = None             # columns to be parsed as datetime
        self.columns_ = None                # original feature columns (for DataFrame creation)

    def fit(self, X, y=None):
        """
        Fit the transformer: store column names, learn label encodings,
        and record the set of one‑hot encoded columns that will be produced.
        """
        X = X.copy()
        self.columns_ = X.columns.tolist()

        # Define columns by type (adjust these lists to match your dataset)
        self.numeric_cols_ = [
            'amount', 'card_id', 'client_id', 'merchant_id',
            'transaction_id', 'num_cards_issued', 'num_credit_cards',
            'yearly_income', 'per_capita_income', 'total_debt',
            'credit_limit', 'current_age', 'cvv'
        ]
        self.date_cols_ = ['date', 'acct_open_date', 'expires', 'year_pin_last_changed']
        self.high_card_cats_ = ['merchant_city', 'merchant_state', 'errors', 'description']
        self.ohe_cats_ = ['use_chip', 'card_brand', 'card_type', 'has_chip', 'generation']

        # Build label encodings from training data (for high‑cardinality columns)
        for col in self.high_card_cats_:
            if col in X.columns:
                # Convert to string and get unique values
                uniques = X[col].astype(str).unique()
                # Create mapping: value -> index (0..n-1)
                mapping = {val: i for i, val in enumerate(uniques)}
                self.label_encoders[col] = mapping

        # Optional: you could also store the columns that will be created by one‑hot encoding
        # by actually applying pd.get_dummies on a subset of training data.
        # Here we'll just remember the category columns; the actual dummy columns will be
        # generated in transform and aligned later (outside this transformer).
        # But for completeness, we could simulate the dummy columns:
        #   temp = X[self.ohe_cats_].copy()
        #   temp = pd.get_dummies(temp, columns=self.ohe_cats_, drop_first=True, dtype=np.int8)
        #   self.ohe_columns_ = temp.columns.tolist()
        # We'll skip for now because alignment is done in _score_dataframe.

        return self

    def transform(self, X):
        """
        Transform the input DataFrame X into a rich feature set.
        - Ensures numeric and date columns have proper dtypes.
        - Drops rows with invalid dates only for rolling calculations (warning emitted).
        - Computes all derived features.
        - Applies label encodings using stored mappings.
        - Adds one‑hot encoded columns.
        """
        # ----- Ensure input is a DataFrame -----
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.columns_)
        else:
            X = X.copy()

        # ----- 1. Coerce numeric columns -----
        for col in self.numeric_cols_:
            if col in X.columns:
                X[col] = pd.to_numeric(X[col], errors='coerce')

        # ----- 2. Coerce date columns -----
        for col in self.date_cols_:
            if col in X.columns:
                X[col] = pd.to_datetime(X[col], errors='coerce')

        # ----- 3. Basic features (that don't rely on groups or dates) -----
        # Absolute amount
        if 'amount' in X.columns:
            X['abs_amount'] = X['amount'].abs().astype('float32')
        else:
            X['abs_amount'] = 0.0

        # Spend to income ratio
        if 'yearly_income' in X.columns and 'amount' in X.columns:
            X['spend_to_income_ratio'] = (
                X['amount'].astype(float) / (X['yearly_income'].astype(float) + 1e-6)
            ).astype('float32')
        else:
            X['spend_to_income_ratio'] = 0.0

        # Time features from date (only if date column exists and has valid values)
        if 'date' in X.columns:
            # Fill NaT with a dummy date? Better to leave as NaT and fill after extraction.
            # But .dt accessor fails on NaT, so we need to handle missing.
            # We'll extract features only for valid dates, fill with 0 afterwards.
            valid_date = X['date'].notna()
            X['hour'] = 0
            X['dayofweek'] = 0
            X['is_weekend'] = 0
            X['is_night'] = 0
            X['is_month_end'] = 0
            if valid_date.any():
                X.loc[valid_date, 'hour'] = X.loc[valid_date, 'date'].dt.hour.astype('int32')
                X.loc[valid_date, 'dayofweek'] = X.loc[valid_date, 'date'].dt.dayofweek.astype('int32')
                X.loc[valid_date, 'is_weekend'] = X.loc[valid_date, 'dayofweek'].isin([5, 6]).astype('int8')
                X.loc[valid_date, 'is_night'] = X.loc[valid_date, 'hour'].between(0, 5).astype('int8')
                X.loc[valid_date, 'is_month_end'] = X.loc[valid_date, 'date'].dt.is_month_end.astype('int8')
        else:
            X['hour'] = 0
            X['dayofweek'] = 0
            X['is_weekend'] = 0
            X['is_night'] = 0
            X['is_month_end'] = 0

        # Account age (if both dates available)
        if 'date' in X.columns and 'acct_open_date' in X.columns:
            age = (X['date'] - X['acct_open_date']).dt.days / 365.0
            X['account_age_years'] = age.fillna(0).astype('float32')
        else:
            X['account_age_years'] = 0.0

        # ----- 4. Sorting for sequential operations -----
        sort_cols = [c for c in ['card_id', 'date'] if c in X.columns]
        if sort_cols:
            # For sorting, we need dates to be datetime; already done.
            X.sort_values(sort_cols, inplace=True)
            X.reset_index(drop=True, inplace=True)

        # ----- 5. Helper functions for expanding statistics -----
        def expanding_mean_shift(arr):
            """Expanding mean, shifted so that current row uses only previous rows."""
            arr = np.asarray(arr, dtype='float64')
            cumsum = np.cumsum(arr)
            count = np.arange(1, len(arr) + 1)
            out = np.empty_like(arr, dtype='float32')
            out[0] = np.nan
            out[1:] = cumsum[:-1] / count[:-1]
            return out

        def expanding_std_shift(arr):
            """Expanding standard deviation (sample), shifted."""
            arr = np.asarray(arr, dtype='float64')
            cumsum = np.cumsum(arr)
            cumsum_sq = np.cumsum(arr ** 2)
            count = np.arange(1, len(arr) + 1)
            out = np.empty_like(arr, dtype='float32')
            out[0] = np.nan
            mean_prev = cumsum[:-1] / count[:-1]
            sqmean_prev = cumsum_sq[:-1] / count[:-1]
            var_prev = np.maximum(sqmean_prev - mean_prev ** 2, 0.0)  # clamp tiny negatives
            out[1:] = np.sqrt(var_prev, dtype='float64').astype('float32')
            return out

        # ----- 6. Card‑level behavior -----
        if {'card_id', 'abs_amount'}.issubset(X.columns):
            X['card_avg_amount'] = (
                X.groupby('card_id')['abs_amount']
                 .transform(expanding_mean_shift)
                 .fillna(0)
                 .astype('float32')
            )
            X['card_std_amount'] = (
                X.groupby('card_id')['abs_amount']
                 .transform(expanding_std_shift)
                 .fillna(0)
                 .astype('float32')
            )
        else:
            X['card_avg_amount'] = 0.0
            X['card_std_amount'] = 0.0

        # ----- 7. Client‑level behavior -----
        if {'client_id', 'abs_amount'}.issubset(X.columns):
            X['client_avg_amount'] = (
                X.groupby('client_id')['abs_amount']
                 .transform(expanding_mean_shift)
                 .fillna(0)
                 .astype('float32')
            )
        else:
            X['client_avg_amount'] = 0.0

        # ----- 8. Z‑score deviation from card average -----
        if 'amount' in X.columns:
            X['amount_zscore_card'] = (
                (X['amount'].astype(float) - X['card_avg_amount']) /
                (X['card_std_amount'] + 1e-6)
            ).astype('float32')
        else:
            X['amount_zscore_card'] = 0.0

        # ----- 9. Merchant repeat features -----
        if {'card_id', 'merchant_id', 'date'}.issubset(X.columns):
            prev_merch = X.groupby('card_id')['merchant_id'].shift(1)
            prev_date = X.groupby('card_id')['date'].shift(1)
            # Time difference in seconds; handle NaT
            time_diff = (X['date'] - prev_date).dt.total_seconds()
            X['rapid_repeat_merchant'] = (
                (X['merchant_id'] == prev_merch) & (time_diff < 300)
            ).fillna(False).astype('int8')
        else:
            X['rapid_repeat_merchant'] = 0

        if {'card_id', 'merchant_id'}.issubset(X.columns):
            X['merchant_txn_count'] = X.groupby(['card_id', 'merchant_id']).cumcount().astype('int32')
            X['new_merchant'] = (X['merchant_txn_count'] == 0).astype('int8')
        else:
            X['merchant_txn_count'] = 0
            X['new_merchant'] = 0

        if 'card_id' in X.columns:
            X['first_occurrence'] = (X.groupby('card_id').cumcount() == 0).astype('int8')
        else:
            X['first_occurrence'] = 0

        # ----- 10. Generation from retirement_age (if present) -----
        if 'retirement_age' in X.columns:
            def get_generation(age):
                try:
                    age = float(age)
                except (TypeError, ValueError):
                    return 'Unknown'
                if 0 <= age <= 25: return 'Gen Z'
                if 26 <= age <= 40: return 'Millennials'
                if 41 <= age <= 56: return 'Gen X'
                if 57 <= age <= 75: return 'Boomers'
                if 76 <= age <= 200: return 'Silent'
                return 'Unknown'
            X['generation'] = X['retirement_age'].apply(get_generation)
        else:
            X['generation'] = 'Unknown'

        # ----- 11. One‑hot encoding of categoricals -----
        # Only columns present in training (self.ohe_cats_) are encoded.
        # To avoid column mismatch, we'll generate dummies and then reindex
        # to the columns seen during fit (if stored). If not stored, we just keep all.
        ohe_cols = [c for c in self.ohe_cats_ if c in X.columns]
        if ohe_cols:
            # Get dummies; this may create new columns
            X_dummies = pd.get_dummies(X[ohe_cols], columns=ohe_cols, drop_first=True, dtype=np.int8)
            # If we have stored dummy columns from fit, align to them
            if self.ohe_columns_ is not None:
                # Add missing columns with 0
                for col in self.ohe_columns_:
                    if col not in X_dummies.columns:
                        X_dummies[col] = 0
                # Keep only the stored columns (in correct order)
                X_dummies = X_dummies[self.ohe_columns_]
            # Drop original categorical columns (they are replaced by dummies)
            X = X.drop(columns=ohe_cols)
            # Concatenate dummies
            X = pd.concat([X, X_dummies], axis=1)
        else:
            # If no OHE columns exist, we still need to ensure any required dummy columns are present.
            # This will be handled later by alignment with model_features.
            pass

        # ----- 12. Apply label encoding for high‑cardinality columns -----
        for col, mapping in self.label_encoders.items():
            if col in X.columns:
                # Convert to string, map, fill unknown with -1
                X[col] = X[col].astype(str).map(mapping).fillna(-1).astype('int32')

        # ----- 13. Rolling 24h transaction count (client‑level) -----
        # This operation requires valid dates; drop rows with invalid dates for this step only.
        if {'client_id', 'date', 'abs_amount'}.issubset(X.columns):
            # Work on a copy to avoid altering the original order
            X_roll = X.copy()
            X_roll['date'] = pd.to_datetime(X_roll['date'], errors='coerce')
            # Drop rows with NaT for the rolling operation
            orig_len = len(X_roll)
            X_roll = X_roll.dropna(subset=['date'])
            if len(X_roll) < orig_len:
                warnings.warn(f"Dropped {orig_len - len(X_roll)} rows with invalid dates for rolling 24h count.")
            # Sort and set index for rolling
            X_roll.sort_values(['client_id', 'date'], inplace=True)
            # Use rolling with '1D' window on date column
            try:
                # groupby then apply rolling
                # We need to ensure the result aligns with original index.
                # We'll compute and then merge back.
                X_roll['txn_last_24h'] = (
                    X_roll.groupby('client_id', group_keys=False)
                          .apply(lambda g: g.rolling('1D', on='date')['abs_amount'].count())
                          .astype('float32')
                )
                # Merge back to original X (preserve rows that were dropped)
                X['txn_last_24h'] = X_roll['txn_last_24h']
                # Fill rows that were dropped (or not computed) with 0
                X['txn_last_24h'] = X['txn_last_24h'].fillna(0).astype('float32')
            except Exception as e:
                # If rolling fails (e.g., due to dtype issues), fallback to 0
                warnings.warn(f"Rolling 24h count failed: {e}. Setting to 0.")
                X['txn_last_24h'] = 0.0
        else:
            X['txn_last_24h'] = 0.0

        # Final: ensure all float columns are float32 and int columns are int32 (for consistency)
        for col in X.select_dtypes(include=['float64']).columns:
            X[col] = X[col].astype('float32')
        for col in X.select_dtypes(include=['int64']).columns:
            X[col] = X[col].astype('int32')

        return X

In [18]:
# --- Custom transformer to select columns ---
class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Keep only the columns that exist in X
        return X[[c for c in self.columns if c in X.columns]]

# --- Final features list ---
final_features = [
    'amount_zscore_card', 'abs_amount', 'client_avg_amount', 'description',
    'spend_to_income_ratio', 'new_merchant', 'zip', 'mcc', 'errors',
    'merchant_state', 'merchant_city', 'current_age', 'first_occurrence',
    'is_weekend', 'txn_last_24h', 'account_age_years',
    'use_chip_Online Transaction', 'use_chip_Swipe Transaction',
    'card_brand_Discovery', 'card_brand_Mastercard', 'card_brand_Visa',
    'card_type_Debit', 'card_type_Prepaid', 'has_chip_YES',
    'generation_Gen X', 'generation_Silent'
]

In [10]:
df_merge['target'] = df_merge['is_fraud'].map({'Yes': 1, 'No': 0})
df_model = df_merge[df_merge['target'].notnull()]
df_target_null = df_merge[df_merge['target'].isnull()]

In [17]:
df_merge['target'].isnull().sum()

np.int64(4243221)

In [30]:
fraud_transactions = df_merge[df_merge['target'] == 1]
print(f"Number of fraud transactions: {len(fraud_transactions)}")

# Display all fraud transactions (or first few)
fraud_transactions.head(10)   # shows first 10 rows

Number of fraud transactions: 12816


,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,birth_month,gender,address,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,is_fraud,target
3459,7479444,2024-01-23 17:20:11,126,5497,3.42,Online Transaction,90999,ONLINE,ONLINE,0.0,...,10,Male,"879 Johannesburg Road, Cape Town",234846,478800,0,799,4,Yes,1.0
9526,7486725,2023-09-16 11:00:05,126,5497,6102.00,Online Transaction,3558,ONLINE,ONLINE,0.0,...,10,Male,"879 Johannesburg Road, Cape Town",234846,478800,0,799,4,Yes,1.0
12989,7490901,2022-01-20 09:26:48,720,4136,80.10,Online Transaction,24504,ONLINE,ONLINE,0.0,...,5,Female,"630 Victoria Road, Johannesburg",174780,356400,720684,682,3,Yes,1.0
13000,7490914,2024-11-10 02:58:49,720,4136,129.42,Online Transaction,38602,ONLINE,ONLINE,0.0,...,5,Female,"630 Victoria Road, Johannesburg",174780,356400,720684,682,3,Yes,1.0
13081,7491008,2023-03-13 06:26:23,1644,3444,157.68,Online Transaction,21776,ONLINE,ONLINE,0.0,...,1,Male,"305 Market Street, Johannesburg",277308,574974,22464,686,6,Yes,1.0
22612,7502435,2024-09-04 15:16:50,126,5497,5229.72,Online Transaction,3558,ONLINE,ONLINE,0.0,...,10,Male,"879 Johannesburg Road, Cape Town",234846,478800,0,799,4,Yes,1.0
22642,7502469,2024-04-12 05:01:41,126,5497,-6102.00,Online Transaction,3558,ONLINE,ONLINE,0.0,...,10,Male,"879 Johannesburg Road, Cape Town",234846,478800,0,799,4,Yes,1.0
25703,7506125,2023-08-23 15:12:33,126,5497,691.38,Online Transaction,59199,ONLINE,ONLINE,0.0,...,10,Male,"879 Johannesburg Road, Cape Town",234846,478800,0,799,4,Yes,1.0
25787,7506219,2024-02-25 22:31:54,1600,5050,3483.72,Online Transaction,60569,ONLINE,ONLINE,0.0,...,9,Female,"29 Nelson Mandela Drive, Gqeberha",893322,1821474,2245878,747,3,Yes,1.0
25942,7506395,2024-05-25 23:59:58,1644,3444,2630.88,Online Transaction,54773,ONLINE,ONLINE,0.0,...,1,Male,"305 Market Street, Johannesburg",277308,574974,22464,686,6,Yes,1.0


In [21]:
df_target_null = df_target_null.drop(columns=["target", "is_fraud"], errors="ignore")

In [23]:
df_target_null.to_csv("df_target_null.csv", index=False)

In [22]:
df_target_null

,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,retirement_age,birth_year,birth_month,gender,address,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
3,7475331,2022-06-12 09:33:41,430,2860,3600.00,Swipe Transaction,27092,Bloemfontein,Free State,9300.0,...,67,1967,5,Female,"573 Durban Road, East London",471024,960300,2316168,685,5
6,7475334,2024-05-06 13:03:32,1556,2972,1386.00,Swipe Transaction,59935,Polokwane,Limpopo,700.0,...,67,1989,7,Female,"405 Pretoria Street, Johannesburg",426222,868986,1982754,740,4
8,7475336,2024-03-21 19:20:35,335,5131,4708.44,Online Transaction,50292,ONLINE,ONLINE,0.0,...,68,1973,7,Female,"641 Jan Smuts Avenue, Cape Town",498528,1016406,1198170,688,3
9,7475337,2022-09-12 15:47:58,351,1112,193.32,Swipe Transaction,3864,Polokwane,Limpopo,700.0,...,70,1928,9,Female,"493 Cape Road, Cape Town",248580,308700,6750,807,6
15,7475343,2023-09-15 00:30:52,1634,2464,19.62,Swipe Transaction,20519,Kimberley,Northern Cape,8300.0,...,68,1953,3,Male,"69 Swart Street, Pietermaritzburg",181638,370386,1082736,825,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12858738,23209451,2023-04-09 09:31:22,1535,187,623.34,Chip Transaction,50524,Durban,KwaZulu-Natal,4000.0,...,64,1960,8,Male,"860 Church Street, Pretoria",294048,599598,1184796,590,2
12858741,23209454,2022-05-18 11:11:19,1603,1313,288.72,Chip Transaction,44211,Johannesburg,Gauteng,2000.0,...,65,1978,12,Male,"472 Durban Road, Cape Town",342648,698616,293958,762,4
12858744,23209457,2023-06-12 07:02:05,461,5482,973.80,Chip Transaction,75936,East London,Eastern Cape,5200.0,...,68,1977,12,Female,"241 Voortrekker Road, Durban",577404,1177254,1441026,784,4
12858750,23209464,2023-07-23 01:05:29,1508,3279,593.46,Chip Transaction,43293,Cape Town,Western Cape,8000.0,...,69,1953,4,Female,"495 Durban Road, Cape Town",344250,701982,1167750,747,4


In [24]:
df_target_null.columns

Index(['transaction_id', 'date', 'client_id', 'card_id', 'amount', 'use_chip',
       'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv',
       'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'description', 'current_age', 'retirement_age',
       'birth_year', 'birth_month', 'gender', 'address', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards'],
      dtype='object')

In [11]:
# --- Prepare X, y ---
X = df_model.drop(columns=['target'])
y = df_model['target']
print(X.shape, y.shape)

(8615533, 35) (8615533,)


In [12]:
from sklearn.model_selection import train_test_split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=False, random_state=45)

In [23]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from collections import Counter

neg, pos = Counter(y_train)[0], Counter(y_train)[1]
spw = neg / pos   

# --- Optimized pipeline ---
pipeline = Pipeline(steps=[
    ("feature_engineering", FraudFeatureEngineer()),
    ("select_features", ColumnSelector(final_features)),
    ("model", XGBClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=spw,
    ))]
)

In [14]:
X_train_final = pipeline.named_steps['feature_engineering'].fit_transform(X_train_raw)
X_train_final = X_train_final[final_features]  # select only numeric/encoded features

In [15]:
X_train_final.columns

Index(['amount_zscore_card', 'abs_amount', 'client_avg_amount', 'description',
       'spend_to_income_ratio', 'new_merchant', 'zip', 'mcc', 'errors',
       'merchant_state', 'merchant_city', 'current_age', 'first_occurrence',
       'is_weekend', 'txn_last_24h', 'account_age_years',
       'use_chip_Online Transaction', 'use_chip_Swipe Transaction',
       'card_brand_Discovery', 'card_brand_Mastercard', 'card_brand_Visa',
       'card_type_Debit', 'card_type_Prepaid', 'has_chip_YES',
       'generation_Gen X', 'generation_Silent'],
      dtype='object')

In [21]:
import joblib
# Load feature engineering pipeline
feat_eng = joblib.load("feature_engineering.pkl")

# Load feature selection pipeline
feat_sel = joblib.load("feature_selection.pkl")

# (Optional) Load the model if you want to test end‑to‑end
model_package = joblib.load("xgb_model_with_threshold.pkl")
model = model_package['model']

In [23]:
raw_df = pd.read_csv("df_target_null.csv")
print("Raw data shape:", raw_df.shape)
print("Raw columns:", list(raw_df.columns))

Raw data shape: (4243221, 34)
Raw columns: ['transaction_id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'description', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']


In [24]:
X_eng = feat_eng.transform(raw_df)
print("After feature engineering - shape:", X_eng.shape)
print("Columns after engineering:", list(X_eng.columns))

After feature engineering - shape: (4243221, 57)
Columns after engineering: ['transaction_id', 'date', 'client_id', 'card_id', 'amount', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'card_number', 'expires', 'cvv', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'description', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'abs_amount', 'spend_to_income_ratio', 'hour', 'dayofweek', 'is_weekend', 'is_night', 'is_month_end', 'account_age_years', 'card_avg_amount', 'card_std_amount', 'client_avg_amount', 'amount_zscore_card', 'rapid_repeat_merchant', 'merchant_txn_count', 'new_merchant', 'first_occurrence', 'use_chip_Online Transaction', 'use_chip_Swipe Transaction', 'card_brand_Discovery', 'card_brand_Mastercard', 'card_brand_Visa', 'card_type_Debit', 'card_type_Prepaid', 'has_chip_YES', 'generation_Gen X',

In [25]:
X_sel = feat_sel.transform(X_eng)
print("After feature selection - shape:", X_sel.shape)
print("Selected columns:", list(X_sel.columns))

After feature selection - shape: (4243221, 26)
Selected columns: ['amount_zscore_card', 'abs_amount', 'client_avg_amount', 'description', 'spend_to_income_ratio', 'new_merchant', 'zip', 'mcc', 'errors', 'merchant_state', 'merchant_city', 'current_age', 'first_occurrence', 'is_weekend', 'txn_last_24h', 'account_age_years', 'use_chip_Online Transaction', 'use_chip_Swipe Transaction', 'card_brand_Discovery', 'card_brand_Mastercard', 'card_brand_Visa', 'card_type_Debit', 'card_type_Prepaid', 'has_chip_YES', 'generation_Gen X', 'generation_Silent']


In [26]:
# If you have the model loaded
model_features = model.get_booster().feature_names
print("Model expects these features:", model_features)
print("Selected features match model? ", set(X_sel.columns) == set(model_features))

Model expects these features: ['amount_zscore_card', 'abs_amount', 'client_avg_amount', 'description', 'spend_to_income_ratio', 'new_merchant', 'zip', 'mcc', 'errors', 'merchant_state', 'merchant_city', 'current_age', 'first_occurrence', 'is_weekend', 'txn_last_24h', 'account_age_years', 'use_chip_Online Transaction', 'use_chip_Swipe Transaction', 'card_brand_Discovery', 'card_brand_Mastercard', 'card_brand_Visa', 'card_type_Debit', 'card_type_Prepaid', 'has_chip_YES', 'generation_Gen X', 'generation_Silent']
Selected features match model?  True


In [27]:
proba = model.predict_proba(X_sel)
print("Prediction probabilities:", proba)

Prediction probabilities: [[9.9941903e-01 5.8094936e-04]
 [9.9963051e-01 3.6949152e-04]
 [9.9949169e-01 5.0831179e-04]
 ...
 [9.9998575e-01 1.4240413e-05]
 [9.9991339e-01 8.6623128e-05]
 [9.9991626e-01 8.3763356e-05]]


In [29]:
import joblib

feature_names = [
    'amount_zscore_card', 'abs_amount', 'client_avg_amount', 'description',
    'spend_to_income_ratio', 'new_merchant', 'zip', 'mcc', 'errors',
    'merchant_state', 'merchant_city', 'current_age', 'first_occurrence',
    'is_weekend', 'txn_last_24h', 'account_age_years',
    'use_chip_Online Transaction', 'use_chip_Swipe Transaction',
    'card_brand_Discovery', 'card_brand_Mastercard', 'card_brand_Visa',
    'card_type_Debit', 'card_type_Prepaid', 'has_chip_YES',
    'generation_Gen X', 'generation_Silent'
]

joblib.dump(feature_names, "model_feature_names.pkl")
print("Saved", len(feature_names), "feature names.")

Saved 26 feature names.


In [54]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from collections import Counter

neg, pos = Counter(y_train)[0], Counter(y_train)[1]
spw = neg / pos   

# --- Optimized pipeline ---
pipeline = Pipeline(steps=[
    ("feature_engineering", FraudFeatureEngineer()),
    ("select_features", ColumnSelector(final_features)),
    ("model", XGBClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=spw,
    ))]
)

In [55]:
pipeline.fit(X_train_raw, y_train)

,steps,"[('feature_engineering', ...), ('select_features', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,columns,"['amount_zscore_card', 'abs_amount', ...]"
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None


In [44]:
from sklearn.metrics import precision_recall_curve, average_precision_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

y_proba = pipeline.predict_proba(X_test_raw)[:, 1]

# PR curve and AP (area under PR)
prec, rec, thr = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
print(f"Average Precision (PR-AUC): {ap:.5f}")

# Example 1: choose threshold for target recall
target_recall = 0.60
idxs = np.where(rec >= target_recall)[0]
thr_rec85 = thr[idxs[-1]] if len(idxs) and idxs[-1] < len(thr) else 0.0
print("Threshold for recall>=0.85:", thr_rec85)

Average Precision (PR-AUC): 0.00122
Threshold for recall>=0.85: 0.001018779


In [45]:
# Example 2: choose threshold that maximizes F2 (recall-weighted)
beta = 2
f2 = (1 + beta**2) * (prec * rec) / (beta**2 * prec + rec + 1e-12)
best_idx = np.nanargmax(f2[:-1])  # thr is len-1 vs prec/rec len
thr_f2 = thr[best_idx]
print("Threshold maximizing F2:", thr_f2)

# Apply a chosen threshold (try thr_f2 first, or pick a small value like 0.01)
chosen_thr = thr_f2
y_pred_custom = (y_proba >= chosen_thr).astype(int)

print(classification_report(y_test, y_pred_custom, digits=4))
print(confusion_matrix(y_test, y_pred_custom))

Threshold maximizing F2: 3.2853462e-05
              precision    recall  f1-score   support

         0.0     0.9997    0.0016    0.0032   2151239
         1.0     0.0012    0.9996    0.0025      2645

    accuracy                         0.0028   2153884
   macro avg     0.5005    0.5006    0.0028   2153884
weighted avg     0.9985    0.0028    0.0032   2153884

[[   3474 2147765]
 [      1    2644]]


In [58]:
from sklearn.metrics import classification_report, confusion_matrix

# For labels
y_pred = pipeline.predict(X_test_raw)

# For probabilities
y_proba = pipeline.predict_proba(X_test_raw)[:, 1]

In [60]:
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# Compute F1 for each threshold
f1_scores = 2 * (precision * recall) / (precision + recall)

# Find the threshold with max F1
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)
print("F1 at best threshold:", f1_scores[best_idx])


Best threshold: 0.7112324
F1 at best threshold: nan


In [61]:
# ===============================
# 6. Threshold
# ===============================
THRESHOLD =  0.7112324
y_proba = (y_pred >= THRESHOLD).astype(int)

In [63]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import auc

# Precision-recall curve (use probabilities)
precision, recall, thresholds = precision_recall_curve(y_test, y_pred)

# PR-AUC
pr_auc = auc(recall, precision)
print("PR-AUC:", pr_auc)

# ROC AUC (use probabilities)
print("ROC AUC:", roc_auc_score(y_test, y_pred))

PR-AUC: 0.020862908785933436
ROC AUC: 0.4999775893863902


In [62]:
print("\nClassification Report:")
print(classification_report(y_test, y_proba, target_names=["Non Fraud", "Fraud"]))


Classification Report:
              precision    recall  f1-score   support

   Non Fraud       1.00      0.96      0.98   2151239
       Fraud       0.00      0.04      0.00      2645

    accuracy                           0.96   2153884
   macro avg       0.50      0.50      0.49   2153884
weighted avg       1.00      0.96      0.98   2153884



In [41]:
import numpy as np
import pandas as pd
from collections import Counter

# 1) Class distribution in y_train and y_test
print("Train distribution:", Counter(y_train))
print("Test distribution :", Counter(y_test))

# 2) What did the model predict?
y_pred = pipeline.predict(X_test_raw)
print("Predicted labels :", Counter(y_pred))

# 3) Are probabilities informative?
y_proba = pipeline.predict_proba(X_test_raw)[:, 1]
print(pd.Series(y_proba).describe())
print("Top 10 probabilities:", np.sort(y_proba)[-10:])

Train distribution: Counter({0.0: 6451478, 1.0: 10171})
Test distribution : Counter({0.0: 2151239, 1.0: 2645})
Predicted labels : Counter({np.int64(0): 2153884})
count    2.153884e+06
mean     1.539227e-03
std      1.710852e-03
min      2.777627e-06
25%      7.708370e-04
50%      1.231326e-03
75%      1.855203e-03
max      2.990299e-01
dtype: float64
Top 10 probabilities: [0.13941857 0.1428519  0.14399707 0.14993101 0.1525255  0.15872113
 0.1945571  0.22778936 0.2434871  0.2990299 ]


In [31]:
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Compute F1 for each threshold
f1_scores = 2 * (precision * recall) / (precision + recall)

# Find the threshold with max F1
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)
print("F1 at best threshold:", f1_scores[best_idx])

Best threshold: 0.08506776
F1 at best threshold: nan


In [35]:
X_test_final.head()

,amount_zscore_card,abs_amount,client_avg_amount,description,spend_to_income_ratio,new_merchant,zip,mcc,errors,merchant_state,...,use_chip_Online Transaction,use_chip_Swipe Transaction,card_brand_Discovery,card_brand_Mastercard,card_brand_Visa,card_type_Debit,card_type_Prepaid,has_chip_YES,generation_Gen X,generation_Silent
320529,5.724000e+09,5724.000000,0.000000,-1,0.005334,1,8000.0,3640.0,-1,-1,...,False,False,False,False,True,True,False,True,False,False
320530,-5.585400e+09,138.600006,5724.000000,-1,0.000129,1,5200.0,5814.0,-1,-1,...,False,False,False,False,True,True,False,True,False,False
1515879,1.159560e+09,1159.560059,824.974365,-1,0.001081,1,700.0,5813.0,-1,-1,...,False,False,False,False,True,False,False,True,False,False
1515880,9.046799e+08,2064.239990,825.245483,-1,0.001924,1,1200.0,5541.0,-1,-1,...,False,False,False,False,True,False,False,True,False,False
320531,-7.094425e-01,950.039978,2931.300049,-1,0.000885,1,300.0,5813.0,-1,-1,...,False,False,False,False,True,True,False,True,False,False


In [32]:
# ===============================
# 6. Threshold
# ===============================
THRESHOLD =  0.19875814
y_proba = (y_pred_proba >= THRESHOLD).astype(int)

In [36]:
# Convert probabilities to 0/1 labels
y_pred_labels = (y_pred_proba >= 0.5).astype(int)

from sklearn.metrics import classification_report

print("\nClassification Report:")
print(classification_report(y_test, y_pred_labels, target_names=["Non Fraud", "Fraud"]))


Classification Report:
              precision    recall  f1-score   support

   Non Fraud       1.00      1.00      1.00   2151239
       Fraud       0.00      0.00      0.00      2645

    accuracy                           1.00   2153884
   macro avg       0.50      0.50      0.50   2153884
weighted avg       1.00      1.00      1.00   2153884



In [37]:
y_test.value_counts()

target
0.0    2151239
1.0       2645
Name: count, dtype: int64

In [24]:
import joblib

# Get the trained feature engineering object
fe = pipeline.named_steps['feature_engineering']

# Save to disk
joblib.dump(fe, "feature_engineering.pkl")

['feature_engineering.pkl']

In [21]:
# Get the trained feature engineering object
select = pipeline.named_steps['select_features']

# Save to disk
joblib.dump(select, "feature_selection.pkl")

['feature_selection.pkl']

In [ ]:
# Save feature engineering pipeline
joblib.dump(feat_eng_pipeline, "feature_engineering.pkl")

# Save feature selection pipeline
joblib.dump(feat_sel_pipeline, "feature_selection.pkl")

In [ ]:
from Feature_Engineering import FraudFeatureEngineer

feat_eng = FraudFeatureEngineer()          # no parameters, or with parameters
feat_eng.fit(X_train_raw)                      # X_train is your raw training data

In [ ]:
from Feature_selection import ColumnSelector

# After feature engineering, you have X_train_eng
selector = ColumnSelector(k=20)            # example parameter
selector.fit(X_train_, y_train)